# Logistic Regression for omics data to classify tumor vs. non-tumor
Author: Inika, 2026-05-06

Used to classify tumor vs. normal samples for methylation data and CNV data

# Load libraries & data

In [4]:
import pandas as pd

In [5]:
cnv = pd.read_csv("data/CNV.csv")
# meth.head

In [6]:
# Read in metadata
metadata = pd.read_csv("data/metadata.csv")
metadata

,patient_id,project,label,sample_type,has_RNASeq,has_DNAm,has_CNV
0,TCGA-AK-3458,TCGA-KIRC,1,Primary Tumor,True,False,True
1,TCGA-B0-5711,TCGA-KIRC,1,Primary Tumor,True,False,True
2,TCGA-B0-5696,TCGA-KIRC,1,Primary Tumor,True,False,True
3,TCGA-CJ-4882,TCGA-KIRC,1,Primary Tumor,True,False,True
4,TCGA-B0-5109,TCGA-KIRC,1,Primary Tumor,True,False,True
...,...,...,...,...,...,...,...
775,TCGA-B0-4713-11A,TCGA-KIRC,0,Solid Tissue Normal,False,True,False
776,TCGA-B0-5094-11A,TCGA-KIRC,0,Solid Tissue Normal,False,True,False
777,TCGA-BP-5198-11A,TCGA-KIRC,0,Solid Tissue Normal,False,True,False
778,TCGA-B0-5100-11A,TCGA-KIRC,0,Solid Tissue Normal,False,True,False


In [7]:
df = cnv

In [8]:
# Use sample names as rownames in metadata
metadata = metadata.set_index("patient_id", drop=False)

# Reformat sample names
df["sample_id"] = df["sample_id"].str.replace(".", "-", regex=False)

def shorten_tcga_id(x):
    
    parts = x.split("-")
    # TCGA barcode structure: sample-level ends at element 4
    # if there are more parts → strip last two
    if len(parts) > 4:
        return "-".join(parts[:4])
    return x


df["sample_id"] = df["sample_id"].apply(shorten_tcga_id)

In [9]:
df.columns

Index(['sample_id', 'FAM41C', 'SAMD11', 'NOC2L', 'KLHL17', 'PLEKHN1', 'HES4',
       'ISG15', 'AGRN', 'C1orf159',
       ...
       'SCO2', 'TYMP', 'ODF3B', 'KLHDC7B', 'CPT1B', 'CHKB', 'MAPK8IP2', 'ARSA',
       'SHANK3', 'ACR'],
      dtype='object', length=14435)

In [10]:
metadata.columns

Index(['patient_id', 'project', 'label', 'sample_type', 'has_RNASeq',
       'has_DNAm', 'has_CNV'],
      dtype='object')

In [11]:
# Subset metadata to only include samples available for this omics
metadata_bool = metadata["patient_id"].isin(df["sample_id"])
metadata_short = metadata[metadata_bool]

In [12]:
print(df.shape)
print(metadata_short.shape)

(349, 14435)
(349, 7)


# Use pre-selected genes

In [20]:
topgenes_multiomics = pd.read_csv("results/top_genes_multiomics.csv", sep = ";")
topgenes_multiomics

,GEX,METH,CNV
0,MIF,cg23097686,FAM174A
1,DGCR5,cg04456219,ST8SIA4
2,EEF1G,cg02326386,SACM1L
3,AQP2,cg01702055,SLCO4C1
4,UBD,cg11201447,SLC25A46
...,...,...,...
495,HMGCR,cg03584506,CCL5
496,ATP6V1A,cg06613738,RDM1
497,MTHFS,cg05471495,MMP28
498,HINT2,cg24390590,GAS2L2


In [22]:
cnv_genes = topgenes_multiomics["CNV"].tolist()
cols = ["sample_id"] + list(df.columns.intersection(cnv_genes))
df = df[cols]
df


,sample_id,TBX15,WARS2,HAO2,HSD3B2,HSD3B1,ZNF697,PHGDH,HMGCS2,REG4,...,PCSK6,TM2D3,TAF15,RASL10B,GAS2L2,MMP28,CCL5,RDM1,DHRS11,MRM1
0,TCGA-AK-3458,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,...,0.495000,0.495000,0.495000,0.495000,0.495000,0.495000,0.495000,0.495000,0.495000,0.495000
1,TCGA-B0-5711,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,...,0.502000,0.502000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000
2,TCGA-B0-5696,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,...,0.501000,0.501000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000
3,TCGA-CJ-4882,0.498000,0.498000,0.498000,0.498000,0.498000,0.498000,0.498000,0.498000,0.498000,...,0.492000,0.492000,0.511000,0.511000,0.511000,0.511000,0.511000,0.511000,0.511000,0.511000
4,TCGA-B0-5109,0.482000,0.482000,0.482000,0.482000,0.482000,0.482000,0.482000,0.482000,0.482000,...,0.482000,0.482000,0.581000,0.581000,0.581000,0.581000,0.581000,0.581000,0.581000,0.581000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344,TCGA-B0-5706-11A,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,...,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597
345,TCGA-CJ-6033-11A,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,...,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597
346,TCGA-CJ-5680-11A,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,...,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597,0.517597
347,TCGA-CZ-5468-11A,0.758906,0.758906,0.758906,0.758906,0.758906,0.758906,0.758906,0.758906,0.758906,...,0.658067,0.658067,0.658067,0.658067,0.658067,0.658067,0.658067,0.658067,0.658067,0.658067


# Split test & training data

In [23]:
# Split training & test data
from sklearn.model_selection import train_test_split

X = df.drop(columns=["sample_id"])
y = metadata_short["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Logistic regression for Methylation data

In [38]:
# Do feature selection and set up the pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("clf", LogisticRegression(
        solver = "saga",
        penalty="elasticnet",
        l1_ratio = 0.75,
        C = 0.75,
        max_iter=100,
        class_weight="balanced"
    ))
])

In [39]:
# Train the model
pipeline.fit(X_train, y_train)

/root/miniforge3/envs/SysCompBio_2026_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/root/miniforge3/envs/SysCompBio_2026_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'elasticnet'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.75
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.75
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=True

In [ ]:
model = pipeline.named_steps["clf"]

importance = pd.Series(
   model.coef_[0],
   index=X_train.columns
).sort_values(key=abs, ascending=False)
print(importance)


SACM1L    1.198753
FOLH1    -0.945594
MRM1     -0.606975
DHRS11   -0.486001
RAG1     -0.414593
            ...   
CXCL12    0.000000
TLE3      0.000000
UACA      0.000000
LARP6     0.000000
GJD4      0.000000
Length: 500, dtype: float64


In [41]:
# Evaluate the model
from sklearn.metrics import classification_report, roc_auc_score

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.69      0.85      0.76        13
           1       0.96      0.91      0.94        57

    accuracy                           0.90        70
   macro avg       0.83      0.88      0.85        70
weighted avg       0.91      0.90      0.90        70

ROC-AUC: 0.8596491228070176
